# 06 - Chunk, embed and push documents to AI Search

## Prerequisites and permissions

- **Existing resources:** an Azure AI Search service with semantic ranker enabled, and a Foundry resource with a `text-embedding-3-small` deployment and available embedding quota. This notebook runs independently of notebook 04.
- **Your identity:** **Search Service Contributor** + **Search Index Data Contributor** on Search, and **Cognitive Services OpenAI User** (or **Foundry Owner**) on the parent Foundry resource. C# calls embeddings as you; project-scoped Foundry User alone does not grant account-level OpenAI inference.
- **Search's system-assigned managed identity:** embedding inference access on the Foundry resource, such as **Cognitive Services OpenAI User** or the broader **Cognitive Services User**, for query-time text vectorization.
- **Not required:** Blob access, an indexer, SharePoint permissions or a tenant administrator role.

**Resource prefix:** set `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). This lab creates only your `alice01-push-chunks` index; chunk/document IDs and vector configuration are scoped inside it. The existing model and Search service are reused. Prefixes prevent naming collisions, not enforce access isolation.

## Overview

**Local text files -> C# chunking -> Azure OpenAI embeddings REST API -> Search documents REST API -> a separate push index**. Unlike notebook 04, this notebook creates no data source, skillset or indexer. The application owns extraction, chunk IDs, chunking, vectorization, retries and deletion. Every request and JSON payload is visible; no Search or OpenAI SDK wrapper is used.

Use the same original synthetic TXT documents, 1,000-character chunks, 200-character overlap, embedding deployment and index schema as notebook 04. This makes the approaches comparable, but a simple character slicer is not guaranteed to match Text Split's sentence-aware boundaries. This lab reads text files only; PDF, DOCX and image extraction are out of scope.

Embedding requests send these synthetic texts to the deployed model and incur usage charges; semantic ranking has separate quota/billing. Use no sensitive data. Your `-push-chunks` index is separate from notebook 04's `-blob-chunks` index.

This notebook is self-contained: BCL/HttpClient helpers are inline below, with no `#!import`, external `.cs` files, SDK packages or infrastructure check. Configuration does not authenticate or make REST calls. Document embeddings run as your signed-in Azure CLI identity; Search's identity embeds query text.

## 1. Configure your lab

Set the required values in the repository-root `.env` and `.env.azure`, including `WORKSHOP_PREFIX`, your Search endpoint and embedding deployment. Run `az login --tenant <AZURE_TENANT_ID>` using the tenant ID from those settings. Select a .NET (C#) kernel with .NET Interactive installed, and run it from the `c-sharp-notebooks` directory so the parent settings and sample-data paths resolve. The cell below loads `.env` first, then `.env.azure`; file values override existing environment variables, and `.env.azure` wins when a setting appears in both.

Keep the model and 1,536 dimensions identical during ingestion and querying. If changing an existing vector field's dimensions, recreate only your own lab index; a schema update alone does not regenerate vectors.

In [1]:
using System.Diagnostics;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Text.RegularExpressions;
using System.Security.Cryptography;
using System;
using System.IO;
using System.Linq;
using System.Net.Http;
using System.Collections.Generic;
using System.Threading.Tasks;
using System.Globalization;
using System.Net;
using System.Threading;


public sealed record PushSettings(string Root, string Tenant, string SearchEndpoint,
    string Account, string Deployment, string Model, string Prefix)
{
    public string Index => Prefix + "-push-chunks";
    public string EmbeddingEndpoint => $"https://{Account}.openai.azure.com";

    public static PushSettings Load(string root)
    {
        LoadEnvFile(Path.Combine(root, ".env"));
        LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string key) => Environment.GetEnvironmentVariable(key)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Missing setting: " + key);
        var settings = new PushSettings(Path.GetFullPath(root), Required("AZURE_TENANT_ID"),
            Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'), Required("AZURE_AI_ACCOUNT_NAME"),
            Required("AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"),
            Environment.GetEnvironmentVariable("AZURE_AI_EMBEDDING_MODEL_NAME")?.Trim() is { Length: > 0 } model
                ? model : "text-embedding-3-small", Required("WORKSHOP_PREFIX"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Regex.IsMatch(Prefix, @"^[a-z][a-z0-9]{2,19}$"))
            throw new InvalidOperationException("Use a tenant GUID and a 3-20 lowercase letter/digit prefix starting with a letter.");
        if (!Uri.TryCreate(SearchEndpoint, UriKind.Absolute, out var uri) || uri.Scheme != "https" ||
            !uri.Host.EndsWith(".search.windows.net", StringComparison.OrdinalIgnoreCase) ||
            uri.Host.Length <= ".search.windows.net".Length || uri.AbsolutePath != "/" ||
            uri.Query != "" || uri.Fragment != "" || uri.UserInfo != "" || !uri.IsDefaultPort)
            throw new InvalidOperationException("Expected a public Azure HTTPS Search endpoint without a path/query.");
        foreach (string value in new[] { Account, Deployment, Model })
            if (!Regex.IsMatch(value, @"^[a-zA-Z0-9][a-zA-Z0-9_.-]{0,127}$"))
                throw new InvalidOperationException("Invalid AI account, deployment or model.");
        if (!Regex.IsMatch(Account, @"^[a-zA-Z0-9][a-zA-Z0-9-]{1,62}$"))
            throw new InvalidOperationException("Invalid AI account name.");
    }

    public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path)) return;
        foreach (string line in File.ReadAllLines(path))
        {
            string setting = line.Trim();
            if (setting.Length == 0 || setting.StartsWith('#')) continue;
            if (setting.StartsWith("export ", StringComparison.Ordinal)) setting = setting[7..].TrimStart();
            int equals = setting.IndexOf('=');
            if (equals <= 0) continue;
            string name = setting[..equals].Trim();
            string value = setting[(equals + 1)..].Trim();
            if (value.Length >= 2 && ((value[0] == '"' && value[^1] == '"') || (value[0] == '\'' && value[^1] == '\'')))
                value = value[1..^1];
            else
            {
                int comment = value.IndexOf(" #", StringComparison.Ordinal);
                if (comment >= 0) value = value[..comment].TrimEnd();
            }
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}

public sealed class PushTransport(HttpClient client, string tenant, Func<string, Task<string>>? tokenProvider = null)
{
    public const string SearchAudience = "https://search.azure.com/";
    public const string EmbeddingAudience = "https://cognitiveservices.azure.com/";
    public const string SearchApiVersion = "2025-09-01";
    public const string EmbeddingApiVersion = "2024-10-21";
    readonly Func<string, Task<string>> getToken = tokenProvider ?? (resource => AzureCliTokenAsync(tenant, resource));

    public Task<JsonObject> SearchAsync(HttpMethod method, string endpoint, string path,
        JsonObject? body, params int[] expected) =>
        SendAsync(method, $"{endpoint}/{path}?api-version={SearchApiVersion}", SearchAudience, body, expected);

    public Task<JsonObject> EmbeddingsAsync(PushSettings settings, JsonObject body) =>
        SendAsync(HttpMethod.Post,
            $"{settings.EmbeddingEndpoint}/openai/deployments/{Uri.EscapeDataString(settings.Deployment)}/embeddings?api-version={EmbeddingApiVersion}",
            EmbeddingAudience, body, [200]);

    async Task<JsonObject> SendAsync(HttpMethod method, string uri, string audience, JsonObject? body, int[] expected)
    {
        using var request = new HttpRequestMessage(method, uri);
        request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await getToken(audience));
        if (body is not null)
            request.Content = new StringContent(body.ToJsonString(), Encoding.UTF8, "application/json");
        using var response = await client.SendAsync(request);
        string text = await response.Content.ReadAsStringAsync();
        if (!expected.Contains((int)response.StatusCode))
        {
            string message = $"HTTP {(int)response.StatusCode} {response.ReasonPhrase}; expected {string.Join(", ", expected)}.";
            Console.Error.WriteLine(message);
            Console.Error.WriteLine("Response body: " + text[..Math.Min(8000, text.Length)]);
            foreach (string name in new[] { "x-ms-request-id", "request-id", "retry-after", "x-ms-retry-after-ms" })
                if (response.Headers.TryGetValues(name, out var values))
                    Console.Error.WriteLine($"{name}: {string.Join(", ", values)}");
            throw new HttpRequestException(message, null, response.StatusCode);
        }
        if ((int)response.StatusCode == 404) return new JsonObject { ["missing"] = true };
        return string.IsNullOrWhiteSpace(text) ? new JsonObject() : JsonNode.Parse(text)!.AsObject();
    }

    public static async Task<string> AzureCliTokenAsync(string tenant, string resource)
    {
        if (!Guid.TryParseExact(tenant, "D", out _) || resource is not (SearchAudience or EmbeddingAudience))
            throw new InvalidOperationException("Invalid Azure CLI tenant or audience.");
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true
        };
        if (OperatingSystem.IsWindows())
            foreach (string arg in new[] { "/d", "/c", "az" }) start.ArgumentList.Add(arg);
        foreach (string arg in new[] { "account", "get-access-token", "--resource", resource, "--tenant", tenant, "--output", "json" })
            start.ArgumentList.Add(arg);
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync();
        Task<string> errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited) process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out. Sign in to the workshop tenant.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed. Run az login --tenant with AZURE_TENANT_ID.");
        return JsonNode.Parse(await output)?["accessToken"]?.GetValue<string>() is { Length: > 0 } token
            ? token : throw new InvalidOperationException("Azure CLI returned no access token.");
    }
}


public sealed class PushWorkshop(PushSettings settings, PushTransport transport, Func<TimeSpan, Task>? pause = null)
{
    public const int Dimensions = 1536;
    public const int ChunkSize = 1000;
    public const int ChunkOverlap = 200;
    public const string SemanticConfiguration = "chunk-semantic";
    public const string Question = "What receipts do I need to claim travel expenses, and when must I submit them?";
    public static readonly string[] QueryModes = ["keyword", "vector", "hybrid", "semantic", "semantic-hybrid"];
    public static readonly string[] Commands = ["index", "chunk", "embed", "push", "verify",
        "keyword", "vector", "hybrid", "semantic", "semantic-hybrid", "run", "cleanup", "help"];
    readonly Func<TimeSpan, Task> delay = pause ?? (duration => Task.Delay(duration));

    public JsonObject IndexPayload() => JsonNode.Parse($$"""
        {
          "name": "{{settings.Index}}",
          "fields": [
            { "name": "chunk_id", "type": "Edm.String", "key": true, "searchable": true, "analyzer": "keyword" },
            { "name": "parent_id", "type": "Edm.String", "filterable": true },
            { "name": "chunk", "type": "Edm.String", "searchable": true },
            { "name": "title", "type": "Edm.String", "searchable": true },
            { "name": "text_vector", "type": "Collection(Edm.Single)", "searchable": true, "retrievable": true,
              "dimensions": {{Dimensions}}, "vectorSearchProfile": "chunk-vector-profile" },
            { "name": "source", "type": "Edm.ComplexType", "fields": [
              { "name": "file_name", "type": "Edm.String", "searchable": true },
              { "name": "url", "type": "Edm.String" }
            ] }
          ],
          "vectorSearch": {
            "algorithms": [ { "name": "chunk-hnsw", "kind": "hnsw", "hnswParameters": { "metric": "cosine" } } ],
            "profiles": [ { "name": "chunk-vector-profile", "algorithm": "chunk-hnsw", "vectorizer": "chunk-vectorizer" } ],
            "vectorizers": [ {
              "name": "chunk-vectorizer", "kind": "azureOpenAI",
              "azureOpenAIParameters": { "resourceUri": "{{settings.EmbeddingEndpoint}}",
                "deploymentId": "{{settings.Deployment}}", "modelName": "{{settings.Model}}" }
            } ]
          },
          "semantic": {
            "defaultConfiguration": "{{SemanticConfiguration}}",
            "configurations": [ {
              "name": "{{SemanticConfiguration}}",
              "prioritizedFields": {
                "titleField": { "fieldName": "title" },
                "prioritizedContentFields": [ { "fieldName": "chunk" } ]
              }
            } ]
          }
        }
        """)!.AsObject();

    public async Task CreateIndexAsync()
    {
        await transport.SearchAsync(HttpMethod.Put, settings.SearchEndpoint, $"indexes/{settings.Index}",
            IndexPayload(), 200, 201, 204);
        Console.WriteLine("Index ready: " + settings.Index);
    }

    public static JsonArray Slice(string text, string fileName, string sourceUrl,
        int chunkSize = ChunkSize, int overlap = ChunkOverlap)
    {
        if (chunkSize <= 0 || overlap < 0 || overlap >= chunkSize)
            throw new ArgumentException("Chunk overlap must be non-negative and smaller than chunk size.");
        if (string.IsNullOrWhiteSpace(text)) throw new ArgumentException("Empty source document: " + fileName);
        string stem = Path.GetFileNameWithoutExtension(fileName);
        if (!Regex.IsMatch(stem, @"^[a-zA-Z0-9_-]+$"))
            throw new ArgumentException("Filename stem must be Search-key-safe.");
        // Python slices Unicode characters, not UTF-16 code units. Normalize TXT newlines like read_text.
        var characters = text.Replace("\r\n", "\n").Replace('\r', '\n').EnumerateRunes().ToArray();
        var documents = new JsonArray();
        for (int start = 0, ordinal = 0; start < characters.Length; start += chunkSize - overlap, ordinal++)
        {
            int end = Math.Min(start + chunkSize, characters.Length);
            string chunk = string.Concat(characters[start..end].Select(rune => rune.ToString()));
            documents.Add(new JsonObject
            {
                ["@search.action"] = "upload",
                ["chunk_id"] = $"{stem}-{ordinal}",
                ["parent_id"] = stem,
                ["title"] = fileName,
                ["chunk"] = chunk,
                ["source"] = new JsonObject { ["file_name"] = fileName, ["url"] = sourceUrl }
            });
            if (end == characters.Length) break;
        }
        return documents;
    }

    public JsonArray ChunkDocuments()
    {
        string directory = Path.Combine(settings.Root, "data", "search");
        var files = Directory.GetFiles(directory, "*.txt").OrderBy(path => path, StringComparer.Ordinal).ToArray();
        if (files.Length != 2) throw new InvalidOperationException("Expected the two workshop sample documents under data/search.");
        var documents = new JsonArray();
        foreach (string path in files)
            foreach (var document in Slice(new UTF8Encoding(false, true).GetString(File.ReadAllBytes(path)), Path.GetFileName(path),
                Path.GetRelativePath(settings.Root, path).Replace('\\', '/')))
                documents.Add(document!.DeepClone());
        Console.WriteLine($"Files: {files.Length} | chunks: {documents.Count}");
        return documents;
    }

    public static void ValidateDocuments(JsonArray documents, bool requireVectors)
    {
        var keys = new HashSet<string>(StringComparer.Ordinal);
        if (documents.Count == 0) throw new InvalidOperationException("Chunk and embed every document before pushing.");
        foreach (var item in documents)
        {
            string? key = item?["chunk_id"]?.GetValue<string>();
            if (key is null || !Regex.IsMatch(key, @"^[a-zA-Z0-9_-]+$") || !keys.Add(key) ||
                item?["@search.action"]?.GetValue<string>() != "upload" ||
                string.IsNullOrWhiteSpace(item?["chunk"]?.GetValue<string>()))
                throw new InvalidOperationException("Invalid or duplicate chunk documents.");
            if (requireVectors) ValidateVector(item!["text_vector"]);
        }
    }

    public static void ValidateVector(JsonNode? node)
    {
        if (node is not JsonArray vector || vector.Count != Dimensions ||
            vector.Any(value => value is null || !double.IsFinite(value.GetValue<double>())))
            throw new InvalidOperationException("Embedding dimensions do not match the Search vector field or contain non-finite values.");
    }

    public async Task EmbedAsync(JsonArray documents)
    {
        foreach (var document in documents) document?.AsObject().Remove("text_vector");
        ValidateDocuments(documents, false);
        try
        {
            for (int start = 0; start < documents.Count; start += 16)
            {
                var batch = documents.Skip(start).Take(16).ToArray();
                var payload = new JsonObject
                {
                    ["input"] = new JsonArray(batch.Select(item => (JsonNode?)JsonValue.Create(item!["chunk"]!.GetValue<string>())).ToArray()),
                    ["dimensions"] = Dimensions
                };
                var response = await transport.EmbeddingsAsync(settings, payload);
                if (response["data"] is not JsonArray embeddings || embeddings.Count != batch.Length)
                    throw new InvalidOperationException("Embedding response did not contain exactly one vector per input.");
                var seen = new HashSet<int>();
                foreach (var embedding in embeddings)
                {
                    if (embedding?["index"] is not JsonValue value || !value.TryGetValue<int>(out int index) ||
                        index < 0 || index >= batch.Length || !seen.Add(index))
                        throw new InvalidOperationException("Embedding response did not contain exactly one vector per input.");
                    ValidateVector(embedding["embedding"]);
                    batch[index]!["text_vector"] = embedding["embedding"]!.DeepClone();
                }
            }
        }
        catch
        {
            foreach (var document in documents) document?.AsObject().Remove("text_vector");
            throw;
        }
        Console.WriteLine("Embedded chunks: " + documents.Count + " | dimensions: " + Dimensions);
    }

    public async Task PushAsync(JsonArray documents)
    {
        ValidateDocuments(documents, true);
        for (int start = 0; start < documents.Count; start += 100)
        {
            var batch = new JsonArray(documents.Skip(start).Take(100).Select(item => item!.DeepClone()).ToArray());
            var response = await transport.SearchAsync(HttpMethod.Post, settings.SearchEndpoint,
                $"indexes/{settings.Index}/docs/index", new JsonObject { ["value"] = batch }, 200, 207);
            var expected = batch.Select(item => item!["chunk_id"]!.GetValue<string>()).Order(StringComparer.Ordinal).ToArray();
            if (response["value"] is not JsonArray results ||
                !results.Select(item => item?["key"]?.GetValue<string>()).Order(StringComparer.Ordinal).SequenceEqual(expected) ||
                results.Any(item => item?["status"]?.GetValue<bool>() != true))
                throw new InvalidOperationException("Push was incomplete or some documents failed: " + response.ToJsonString());
            Console.WriteLine("Uploaded chunks: " + batch.Count);
        }
    }

    public async Task VerifyAsync(JsonArray documents)
    {
        ValidateDocuments(documents, true);
        var pending = new List<string>();
        for (int attempt = 0; attempt < 30; attempt++)
        {
            pending.Clear();
            foreach (var document in documents)
            {
                string key = document!["chunk_id"]!.GetValue<string>();
                var stored = await transport.SearchAsync(HttpMethod.Get, settings.SearchEndpoint,
                    $"indexes/{settings.Index}/docs/{Uri.EscapeDataString(key)}", null, 200, 404);
                bool match = stored["missing"]?.GetValue<bool>() != true &&
                    stored["chunk"]?.GetValue<string>() == document["chunk"]!.GetValue<string>() &&
                    stored["text_vector"] is JsonArray { Count: Dimensions } actual &&
                    actual.Zip(document["text_vector"]!.AsArray()).All(pair =>
                        pair.First is not null && double.IsFinite(pair.First.GetValue<double>()) &&
                        Math.Abs(pair.First.GetValue<double>() - pair.Second!.GetValue<double>()) <= 1e-6);
                if (!match) pending.Add(key);
            }
            if (pending.Count == 0)
            {
                foreach (var document in documents)
                    Console.WriteLine($"{document!["chunk_id"]} | vector dimensions: {Dimensions}");
                return;
            }
            if (attempt < 29) await delay(TimeSpan.FromSeconds(2));
        }
        throw new TimeoutException("Pushed documents are not yet visible with the expected content: " + string.Join(", ", pending));
    }

    public static JsonObject QueryPayload(string mode, string question)
    {
        if (!QueryModes.Contains(mode)) throw new ArgumentException("Unknown query mode.");
        if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
        var payload = new JsonObject { ["select"] = "chunk_id,title,chunk,source", ["top"] = 3 };
        if (mode != "vector")
        {
            payload["search"] = question;
            payload["searchFields"] = "chunk";
        }
        if (mode is "vector" or "hybrid" or "semantic-hybrid")
            payload["vectorQueries"] = new JsonArray(new JsonObject
            {
                ["kind"] = "text",
                ["text"] = question,
                ["fields"] = "text_vector",
                ["k"] = 50
            });
        if (mode is "semantic" or "semantic-hybrid")
        {
            payload["queryType"] = "semantic";
            payload["semanticConfiguration"] = SemanticConfiguration;
            payload["captions"] = "extractive";
        }
        return payload;
    }

    public async Task QueryAsync(string mode, string question)
    {
        var response = await transport.SearchAsync(HttpMethod.Post, settings.SearchEndpoint,
            $"indexes/{settings.Index}/docs/search", QueryPayload(mode, question), 200);
        if (response["@search.semanticPartialResponseReason"]?.ToString() is { Length: > 0 })
            throw new InvalidOperationException("Semantic ranking did not complete: " + response.ToJsonString());
        Console.WriteLine(response.ToJsonString());
    }

    public async Task CleanupAsync(bool confirm)
    {
        if (!confirm) throw new InvalidOperationException("Cleanup disabled. Use cleanup --confirm to delete only your push index.");
        settings.Validate();
        await transport.SearchAsync(HttpMethod.Delete, settings.SearchEndpoint,
            $"indexes/{settings.Index}", null, 200, 204, 404);
        Console.WriteLine("Deleted or already absent: " + settings.Index);
    }

}
string root = Path.GetFullPath("..");
PushSettings settings = PushSettings.Load(root);
HttpClient client = new HttpClient { Timeout = TimeSpan.FromSeconds(90) };
Func<string, Task<string>> tokenProvider = resource => PushTransport.AzureCliTokenAsync(settings.Tenant, resource);
Func<TimeSpan, Task> pause = duration => Task.Delay(duration);
PushTransport Transport() => new PushTransport(client, settings.Tenant, tokenProvider);
PushWorkshop Lab() => new PushWorkshop(settings, Transport(), pause);
JsonArray documents = new JsonArray();
bool uploaded = false;
Console.WriteLine("Push index: " + settings.Index);
Console.WriteLine("Embedding deployment: " + settings.Deployment);

Push index: dib01-push-chunks
Embedding deployment: workshop-embedding


## 2. Create the separate push index

The schema mirrors notebook 04: chunk text, filename, provenance and a vector field. Here **C#**, not index projections, supplies `chunk_id` and `parent_id`. HNSW uses cosine similarity. The keyless vectorizer lets Search embed query text using its own managed identity. Semantic ranking prioritizes `title` and `chunk`.

In [2]:
JsonObject index = JsonNode.Parse($$"""
        {
          "name": "{{settings.Index}}",
          "fields": [
            { "name": "chunk_id", "type": "Edm.String", "key": true, "searchable": true, "analyzer": "keyword" },
            { "name": "parent_id", "type": "Edm.String", "filterable": true },
            { "name": "chunk", "type": "Edm.String", "searchable": true },
            { "name": "title", "type": "Edm.String", "searchable": true },
            { "name": "text_vector", "type": "Collection(Edm.Single)", "searchable": true, "retrievable": true,
              "dimensions": {{PushWorkshop.Dimensions}}, "vectorSearchProfile": "chunk-vector-profile" },
            { "name": "source", "type": "Edm.ComplexType", "fields": [
              { "name": "file_name", "type": "Edm.String", "searchable": true },
              { "name": "url", "type": "Edm.String" }
            ] }
          ],
          "vectorSearch": {
            "algorithms": [ { "name": "chunk-hnsw", "kind": "hnsw", "hnswParameters": { "metric": "cosine" } } ],
            "profiles": [ { "name": "chunk-vector-profile", "algorithm": "chunk-hnsw", "vectorizer": "chunk-vectorizer" } ],
            "vectorizers": [ {
              "name": "chunk-vectorizer", "kind": "azureOpenAI",
              "azureOpenAIParameters": { "resourceUri": "{{settings.EmbeddingEndpoint}}",
                "deploymentId": "{{settings.Deployment}}", "modelName": "{{settings.Model}}" }
            } ]
          },
          "semantic": {
            "defaultConfiguration": "{{PushWorkshop.SemanticConfiguration}}",
            "configurations": [ {
              "name": "{{PushWorkshop.SemanticConfiguration}}",
              "prioritizedFields": {
                "titleField": { "fieldName": "title" },
                "prioritizedContentFields": [ { "fieldName": "chunk" } ]
              }
            } ]
          }
        }
        """)!.AsObject();
await Transport().SearchAsync(HttpMethod.Put, settings.SearchEndpoint,
    $"indexes/{settings.Index}", index, 200, 201, 204);
Console.WriteLine("Index ready: " + settings.Index);

Index ready: dib01-push-chunks


## 3. Read and chunk the source files in C#

Use a visible character-slicing loop: at most 1,000 characters, with 200 characters of overlap. These are characters, not tokens or physical pages. The original filenames contain only Search-key-safe letters and hyphens; their stems and chunk numbers give deterministic keys for repeat uploads. `source.url` is a repository-relative path, not a Blob URL or a promise of hosted access.

Rerunning with unchanged inputs replaces documents at the same keys. **Shortening/removing files or changing chunking can leave old keys behind.** Before those experiments, use the opt-in cleanup to delete only this push index, then rerun from section 2. A production application must track and delete obsolete chunks explicitly.

In [3]:
uploaded = false;
documents = new JsonArray(); // Clear old chunks/vectors before reading changed inputs.
documents = Lab().ChunkDocuments();
Console.WriteLine(documents[0].ToJsonString());

Files: 2 | chunks: 6
{"@search.action":"upload","chunk_id":"onboarding-guide-0","parent_id":"onboarding-guide","title":"onboarding-guide.txt","chunk":"Northwind Workshop - Synthetic Onboarding Guide\n\nThis fictional document is provided for an Azure AI Search workshop. It is not guidance for any real employer and contains no personal or confidential information.\n\nBefore the first workshop session\nParticipants receive a schedule, a link to the learning material and a list of local prerequisites. Install a supported Python interpreter, Visual Studio Code, the Jupyter extension and Azure CLI. Sign in to the workshop tenant with an approved account. The facilitator grants access to the shared learning resources. Participants should not request subscription-wide administrator privileges to complete a notebook.\n\nWorkspace setup\nCreate a local virtual environment and install the workshop requirements. Select that environment as the Jupyter kernel. Read the environment example before ad

## 4. Generate embeddings through the REST API

Here the **signed-in user**, not Search, calls Azure OpenAI. Requests contain small batches of up to 16 chunks. The deployment name belongs in the URL; `modelName` in the index vectorizer is the underlying model name, not necessarily the deployment name. Each embedding is matched to its input using the response's `index`, then checked against the vector field's dimensions.

A 403 usually means missing account-level inference access; a 429 can indicate embedding quota. Errors are surfaced rather than silently skipped. No embeddings are regenerated inside the following push cell.

In [4]:
uploaded = false;
// Inline EmbedAsync above shows the input/dimensions JSON and batches of 16.
// It maps by returned index and clears every vector if any request/validation fails.
await Lab().EmbedAsync(documents);

Embedded chunks: 6 | dimensions: 1536


## 5. Push enriched chunks to Search

`POST /indexes/{name}/docs/index` receives the text, vectors, keys and provenance with `@search.action='upload'`. Small batches of 100 fit this lab's 1,536-dimension vectors comfortably within Search's 1,000-document/16-MB limits. For larger real-world inputs, also enforce the encoded request size.

A 207 response can mean some documents failed. Check every returned document's `status`, not just the HTTP status. A failed batch may already have written some documents: correct the error and rerun the push with the same keys; do not generate new keys to hide failures. No automated retry loop is added.

In [5]:
uploaded = false;
// Inline PushAsync above sends {"value": batch} with @search.action="upload".
// It validates all vectors first, batches 100, and checks every key/status even at HTTP 207.
await Lab().PushAsync(documents);
uploaded = true;

Uploaded chunks: 6


## 6. Verify ingestion

Indexing is asynchronous. Poll each uploaded key until its stored text and vector match the document we sent, rather than accepting a successful batch response as proof that search is ready. This also avoids mistaking an older document at the same key for the new content. Print only IDs and vector lengths, not full vectors.

In [6]:
if (!uploaded) throw new InvalidOperationException("Push every document successfully before verification.");
// GET every uploaded key, comparing actual text and vectors, not just vector lengths.
await Lab().VerifyAsync(documents);

onboarding-guide-0 | vector dimensions: 1536
onboarding-guide-1 | vector dimensions: 1536
onboarding-guide-2 | vector dimensions: 1536
travel-policy-0 | vector dimensions: 1536
travel-policy-1 | vector dimensions: 1536
travel-policy-2 | vector dimensions: 1536


## 7. Compare keyword, vector, hybrid and semantic-ranked search

Use the same question and `top=3` as notebook 04. Keyword uses BM25, vector uses embedding similarity, and hybrid fuses their ranks with RRF. Semantic ranking reranks candidates and returns extractive captions; it is not answer generation. Compare sources and ordering, not raw scores between modes. The small dataset may yield identical rankings.

During ingestion C# generated vectors. Below, `kind='text'` delegates **query-time** embeddings to the Search vectorizer and Search's identity. This does not turn push ingestion into an indexer pipeline.

Results are pretty-printed as indented JSON using `JsonSerializerOptions { WriteIndented = true }`, making individual hits, scores, captions and source fields easier to inspect without changing the returned data.

### Keyword search

In [7]:
string question = PushWorkshop.Question;
if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
JsonObject keyword_payload = new JsonObject
{
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3,
    ["search"] = question, ["searchFields"] = "chunk",
};
JsonObject keyword_results = await Transport().SearchAsync(HttpMethod.Post, settings.SearchEndpoint,
    $"indexes/{settings.Index}/docs/search", keyword_payload, 200);
if (keyword_results["@search.semanticPartialResponseReason"]?.ToString() is { Length: > 0 })
    throw new InvalidOperationException("Semantic ranking did not complete: " + keyword_results.ToJsonString());
Console.WriteLine(keyword_results.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-push-chunks\u0027)/$metadata#docs(*)",
  "value": [
    {
      "@search.score": 2.0811255,
      "chunk_id": "travel-policy-0",
      "chunk": "Northwind Workshop - Synthetic Travel Policy\n\nThis document is fictional workshop material. It contains no real employee records or corporate policy. Use it to explore indexing, chunk boundaries, provenance and keyword retrieval.\n\nTravel planning\nEmployees should discuss the purpose, expected outcomes and estimated cost of a trip with their manager before booking. The fictional Northwind Workshop organization prefers remote meetings when they achieve the same result. A trip request includes destination, travel dates, transport options and a short explanation of the customer or training activity. Manager approval should be recorded before any nonrefundable booking.\n\nTravel reimbursement and receipts\nSubmit travel reimbursement r

### Vector-only search

No `search` property: there are no keyword candidates in this request.

In [8]:
if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
JsonObject vector_payload = new JsonObject
{
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3,
    ["vectorQueries"] = new JsonArray(new JsonObject
    { ["kind"] = "text", ["text"] = question, ["fields"] = "text_vector", ["k"] = 50 }),
};
JsonObject vector_results = await Transport().SearchAsync(HttpMethod.Post, settings.SearchEndpoint,
    $"indexes/{settings.Index}/docs/search", vector_payload, 200);
if (vector_results["@search.semanticPartialResponseReason"]?.ToString() is { Length: > 0 })
    throw new InvalidOperationException("Semantic ranking did not complete: " + vector_results.ToJsonString());
Console.WriteLine(vector_results.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-push-chunks\u0027)/$metadata#docs(*)",
  "value": [
    {
      "@search.score": 0.7093305,
      "chunk_id": "travel-policy-1",
      "chunk": " an itemized receipt for lodging, transport and any individual expense greater than twenty-five fictional credits. A card statement by itself is not an itemized receipt. If a receipt is missing, include a written explanation and request an exception from the finance reviewer. Duplicate claims are not permitted. Personal entertainment and unrelated shopping are not reimbursable.\n\nHotels and transport\nSelect a reasonably priced hotel near the meeting location. Public transit is encouraged when practical. If a rental car is necessary, record why public transit was not suitable and include the rental agreement with the expense report. Combining a personal vacation with a workshop trip requires a clear separation of personal costs. The f

### Hybrid search (keyword + vector)

In [9]:
if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
JsonObject hybrid_payload = new JsonObject
{
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3,
    ["search"] = question, ["searchFields"] = "chunk",
    ["vectorQueries"] = new JsonArray(new JsonObject
    { ["kind"] = "text", ["text"] = question, ["fields"] = "text_vector", ["k"] = 50 }),
};
JsonObject hybrid_results = await Transport().SearchAsync(HttpMethod.Post, settings.SearchEndpoint,
    $"indexes/{settings.Index}/docs/search", hybrid_payload, 200);
if (hybrid_results["@search.semanticPartialResponseReason"]?.ToString() is { Length: > 0 })
    throw new InvalidOperationException("Semantic ranking did not complete: " + hybrid_results.ToJsonString());
Console.WriteLine(hybrid_results.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-push-chunks\u0027)/$metadata#docs(*)",
  "value": [
    {
      "@search.score": 0.03306011110544205,
      "chunk_id": "travel-policy-1",
      "chunk": " an itemized receipt for lodging, transport and any individual expense greater than twenty-five fictional credits. A card statement by itself is not an itemized receipt. If a receipt is missing, include a written explanation and request an exception from the finance reviewer. Duplicate claims are not permitted. Personal entertainment and unrelated shopping are not reimbursable.\n\nHotels and transport\nSelect a reasonably priced hotel near the meeting location. Public transit is encouraged when practical. If a rental car is necessary, record why public transit was not suitable and include the rental agreement with the expense report. Combining a personal vacation with a workshop trip requires a clear separation of personal co

### Keyword search + semantic ranking

Inspect `@search.rerankerScore` and `@search.captions`. Do not present a partial response as successful reranking.

In [10]:
if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
JsonObject semantic_payload = new JsonObject
{
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3,
    ["search"] = question, ["searchFields"] = "chunk",
    ["queryType"] = "semantic", ["semanticConfiguration"] = "chunk-semantic",
    ["captions"] = "extractive",
};
JsonObject semantic_results = await Transport().SearchAsync(HttpMethod.Post, settings.SearchEndpoint,
    $"indexes/{settings.Index}/docs/search", semantic_payload, 200);
if (semantic_results["@search.semanticPartialResponseReason"]?.ToString() is { Length: > 0 })
    throw new InvalidOperationException("Semantic ranking did not complete: " + semantic_results.ToJsonString());
Console.WriteLine(semantic_results.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-push-chunks\u0027)/$metadata#docs(*)",
  "@search.answers": [],
  "value": [
    {
      "@search.score": 2.0811255,
      "@search.rerankerScore": 2.537198066711426,
      "@search.captions": [
        {
          "text": "Northwind Workshop - Synthetic Travel Policy  This document is fictional workshop material. It contains no real employee records or corpora... Manager approval should be recorded before any nonrefundable booking. Travel reimbursement and receipts Submit travel reimbursement requests within thirty calendar days after the trip ends. Attach an.",
          "highlights": "Northwind Workshop - Synthetic Travel Policy  This document is fictional workshop material. It contains no real employee records or corpora... Manager approval should be recorded before any nonrefundable booking. Travel reimbursement and\u003Cem\u003E receipts Submit travel reimbursement reques

### Hybrid search + semantic ranking

`k=50` supplies up to 50 vector candidates before reranking; the sample corpus contains fewer.

In [11]:
if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question.");
JsonObject semantic_hybrid_payload = new JsonObject
{
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3,
    ["search"] = question, ["searchFields"] = "chunk",
    ["vectorQueries"] = new JsonArray(new JsonObject
    { ["kind"] = "text", ["text"] = question, ["fields"] = "text_vector", ["k"] = 50 }),
    ["queryType"] = "semantic", ["semanticConfiguration"] = "chunk-semantic",
    ["captions"] = "extractive",
};
JsonObject semantic_hybrid_results = await Transport().SearchAsync(HttpMethod.Post, settings.SearchEndpoint,
    $"indexes/{settings.Index}/docs/search", semantic_hybrid_payload, 200);
if (semantic_hybrid_results["@search.semanticPartialResponseReason"]?.ToString() is { Length: > 0 })
    throw new InvalidOperationException("Semantic ranking did not complete: " + semantic_hybrid_results.ToJsonString());
Console.WriteLine(semantic_hybrid_results.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-push-chunks\u0027)/$metadata#docs(*)",
  "@search.answers": [],
  "value": [
    {
      "@search.score": 0.03279569745063782,
      "@search.rerankerScore": 2.537198066711426,
      "@search.captions": [
        {
          "text": "Northwind Workshop - Synthetic Travel Policy  This document is fictional workshop material. It contains no real employee records or corpora... Manager approval should be recorded before any nonrefundable booking. Travel reimbursement and receipts Submit travel reimbursement requests within thirty calendar days after the trip ends. Attach an.",
          "highlights": "Northwind Workshop - Synthetic Travel Policy  This document is fictional workshop material. It contains no real employee records or corpora... Manager approval should be recorded before any nonrefundable booking. Travel reimbursement and\u003Cem\u003E receipts Submit travel reimbursem

## 8. Compare ingestion approaches

| Responsibility | 04: Blob indexer | 06: Push API |
|---|---|---|
| Read/extract text | Search reads Blob files | C# reads local TXT files |
| Chunk documents | Text Split skill | Visible C# slicing loop |
| Generate document embeddings | Embedding skill, Search identity | Embeddings REST call, user identity |
| Assign keys and write chunks | Index projections + indexer | C# + documents REST API |
| Detect changes and remove old chunks | Indexer change tracking; configure deletion detection | Application tracks changes/deletions |
| Embed query text | Search vectorizer | Search vectorizer |
| Retrieve/rerank | Keyword, vector, hybrid, semantic | Same search features, separate index |

Compare chunk counts and boundaries, sources and query results in the two indexes. Explain why adding a vectorizer alone does not generate document vectors, and why semantic ranking is distinct from vector retrieval. Experiment with smaller chunks only after deleting/recreating this push index to avoid stale keys.

## 9. Optional cleanup

Delete only your named `-push-chunks` index. This never deletes notebook 04's index, any blobs, the shared Search service or model deployment. After cleanup, rerun from section 2 to rebuild.

In [ ]:
bool CLEAN_UP = false;
if (CLEAN_UP)
    await Lab().CleanupAsync(true); // Exact validated prefixed push index only; 404 is okay.
else
    Console.WriteLine("Cleanup disabled.");

## References

- [REST index/skillset style reference](https://github.com/memasanz/mm-ai-search/blob/main/000_CreateIndex.ipynb)
- [Generate embeddings](https://learn.microsoft.com/azure/search/vector-search-how-to-generate-embeddings)
- [Load an index with the push APIs](https://learn.microsoft.com/azure/search/search-how-to-load-search-index)
- [Azure OpenAI vectorizer](https://learn.microsoft.com/azure/search/vector-search-vectorizer-azure-open-ai)
- [Hybrid search](https://learn.microsoft.com/azure/search/hybrid-search-overview)
- [Semantic ranking](https://learn.microsoft.com/azure/search/semantic-search-overview)

Next: SharePoint indexed ingestion with document-level permissions in notebook 07.